In [ ]:
"""
Created on: 18-08-2026

@author B.A. Sturre

PyTorch tutorial 
https://www.youtube.com/watch?v=c36lUUr864M 
"""

import torch

## Augograd Gradient Computation

In [11]:
# make tensor with requires_grad=True
x = torch.randn(3, requires_grad=True)
print(x)

tensor([ 0.4466, -0.1108,  1.2092], requires_grad=True)


Pytorch will create an operational graph for each operation, with input (i.e. x and 2), an operation (i.e. +) and an output (i.e. y). With this graph and back propagation we can calculate the gradient: first we do a forward pass (y=x+2) and since we clarified that we want to calculate the gradient, PyTorch will automatically create and store a function for us (`grad_fn`), which is then use in the back propagation to get the gradient. 

In [12]:
y = x+2
print(y)

z = y*y*2 
print(z)

z = z.mean()
print(z)

tensor([2.4466, 1.8892, 3.2092], grad_fn=<AddBackward0>)
tensor([11.9715,  7.1378, 20.5977], grad_fn=<MulBackward0>)
tensor(13.2357, grad_fn=<MeanBackward0>)


calculating the gradients of z w.r.t. x.

In the background this will create a backward Jacobian matrix (Jacobian multiplied by gradient vector). 

In [13]:
z.backward()  # dz/dx
print(x.grad)

tensor([3.2621, 2.5189, 4.2789])


if z is a vector (i.e. we didn't apply the mean), we need an argument in `z.backward()`. We need for this a gradient input of a tensor of the same size.

In [14]:
y = x+2
z = y*y*2 

v = torch.tensor([0.1, 1.0, 0.001], dtype=torch.float32)
z.backward(v)  # dz/dx
print(x.grad)

tensor([ 4.2407, 10.0755,  4.2917])


Preventing PyTorch from tracking history and calculating `grad_fn` (for example during our training when we want to update our weight this operation should not be part of the gradient operation)

In [15]:
# option 1: 
x = torch.randn(3, requires_grad=True)
print(x)
x.requires_grad_(False)  # trailing underscore: modify variable in place
print(x, '\n')

# option 2 
x = torch.randn(3, requires_grad=True)
print(x)
y = x.detach()  # create a new tensor that doesn't require the gradient
print(y, '\n')

# option 3
x = torch.randn(3, requires_grad=True)
print(x)
with torch.no_grad():
    # operations
    y = x + 2
    print(y)

tensor([-1.1309, -0.2010,  1.8406], requires_grad=True)
tensor([-1.1309, -0.2010,  1.8406]) 

tensor([ 0.2316,  0.1354, -0.8271], requires_grad=True)
tensor([ 0.2316,  0.1354, -0.8271]) 

tensor([-0.8654,  0.3822, -0.2664], requires_grad=True)
tensor([1.1346, 2.3822, 1.7336])


Whenever we call the `.backward()` function the gradient for the tensor will be accumulated into the `.grad()` attribute, so the values will be summed up, so we must be very careful

In [16]:
weights = torch.ones(4, requires_grad=True)

# mock training loop
for epoch in range(3):
    # dummy model
    model_output = (weights * 3).sum()

    # calculating the gradient
    model_output.backward()

    # all the values are summed up so our weights/gradients are after step 1
    # clearly incorrect
    print(weights.grad)

tensor([3., 3., 3., 3.])
tensor([6., 6., 6., 6.])
tensor([9., 9., 9., 9.])


To solve this we must empty the gradient

In [17]:
weights = torch.ones(4, requires_grad=True)

# mock training loop
for epoch in range(3):
    # dummy model
    model_output = (weights * 3).sum()

    # calculating the gradient
    model_output.backward()

    print(weights.grad)

    # empty the gradient
    weights.grad.zero_()

tensor([3., 3., 3., 3.])
tensor([3., 3., 3., 3.])
tensor([3., 3., 3., 3.])


Later we will do this with the PyTorch built-in optimizer 

In [19]:
weights = torch.ones(4, requires_grad=True)

# choose optimizer (SGD: Stochastic Gradient Descent)
optimizer = torch.optim.SGD([weights], lr=0.01) # lr = learning rate

# doing an optimization step 
optimizer.step()

# emtpy the gradient (same utility as the .zero_())
optimizer.zero_grad()